## Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output

- 1 Pydantic
- 2 TypedDist
- 3 Data Classes

### <u>Pydantic</u>

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [41]:
import os
from langchain.chat_models import init_chat_model

model=init_chat_model("ollama:llama3.2")
model


ChatOllama(metadata={'lc_versions': {'langchain-core': '1.7.3', 'langchain': '1.4.7'}}, model='llama3.2')

In [30]:
from pydantic import BaseModel, Field

# creating the template for the structured output
class Movie(BaseModel):
    title:str=Field(description="The title of the movie"),
    year:int= Field(description="This year the movie was released"),
    director:str= Field(description="Director of the movie"),
    rating:float=Field(description="The rating of the movie")   


In [36]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

/Users/amitsah/Projects/learn-agentic-ai/.venv/lib/python3.14/site-packages/pydantic/json_schema.py:2671: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The title of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/Users/amitsah/Projects/learn-agentic-ai/.venv/lib/python3.14/site-packages/pydantic/json_schema.py:2671: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='This year the movie was released'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/Users/amitsah/Projects/learn-agentic-ai/.venv/lib/python3.14/site-packages/pydantic/json_schema.py:2671: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='Director of the movie'),) is not JS

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.7.3', 'langchain': '1.4.7', 'langchain-openai': '1.7.3'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x11ae8a190>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x11ae8a520>, root_client=<openai.OpenAI object at 0x11ae89e

In [44]:
# the vague response
response = model.invoke("Provide me the detail for the movie Marvel") 
response

KeyboardInterrupt: 

In [40]:
# the structured response
model_with_structure.invoke("Provide me the detail about movie 1920")

/Users/amitsah/Projects/learn-agentic-ai/.venv/lib/python3.14/site-packages/pydantic/json_schema.py:2671: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The title of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/Users/amitsah/Projects/learn-agentic-ai/.venv/lib/python3.14/site-packages/pydantic/json_schema.py:2671: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='This year the movie was released'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/Users/amitsah/Projects/learn-agentic-ai/.venv/lib/python3.14/site-packages/pydantic/json_schema.py:2671: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='Director of the movie'),) is not JS

OpenAIAuthenticationError: Error code: 401 - Incorrect API key provided: OPENAI_A**_KEY. You can find your API key at https://platform.openai.com/account/api-keys.

## Nested Structure in Pydantic

In [45]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str
    
class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    
model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Movie Avatar")
response

MovieDetails(title='Avatar', year=2009, cast=[Actor(name='Sam Worthington', role='Jake Sully'), Actor(name='Zoe Saldana', role='Neytiri'), Actor(name='Sigourney Weaver', role='Dr. Norma Spellman'), Actor(name='Michelle Rodriguez', role='Trudy Chacón'), Actor(name='Giovanni Ribisi', role='Parker Selfridge')])